<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">《从零构建推理模型》</a>一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 附录 E：批处理与面向吞吐量的执行

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # 用于下载函数
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

- 正文各章通常一次处理一个样本
- 这样能让代码更紧凑、更容易理解
- 此外，这些代码的运行成本本就很高，受硬件和资源限制，加入批处理支持带来的收益有限
- 不过，在某些场景下，以批处理模式运行代码仍然很有用
- 本附录介绍批处理执行的总体思路，并展示如何利用附加材料中的代码，将其应用于不同章节


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F01_raschka.webp" width="400px">

&nbsp;
## E.1 批处理为何有帮助

- 有两个不同的性能目标：
  - 延迟：获得单个提示词答案的速度；
  - 吞吐量：给定时间内可以处理的提示词数量。
- 单样本生成通常最适合降低延迟和调试代码
- 批处理主要面向吞吐量
- 如果要在 MATH-500 上评估数百个问题、生成大量自洽样本，或在许多监督样本上训练，合适硬件上的批处理可以大幅缩短总运行时间
  - 不过，批处理并不能保证在所有设备上都更快
  - CPU 上的小模型或某些优化不足的 GPU 可能无法受益，甚至会变慢，因为额外的填充和批处理开销可能抵消并行带来的收益


&nbsp;
## E.2 运行批量生成

- 批处理的主要技术障碍是提示词长度通常不同
- 例如，一个数学问题分词后可能有 40 个词元，另一个则可能有 120 个
- PyTorch 张量必须为矩形，因此需要填充较短的序列，使它们都能放入同一个批次张量


- 从概念上说，这使批量生成比单提示词生成更难实现
- 正文章节使用 `reasoning_from_scratch.qwen3` 中的 `Qwen3Model` 类（它采用附录 C 介绍的 Qwen3 实现）
- 批量生成需要跟踪填充词元等信息，因此 `reasoning_from_scratch.qwen3_batched` 中另有一个 `Qwen3Model` 类（源代码见附加材料：https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3_batched.py）


- 为说明批量生成工具的用法，下面查看一个具体示例
- 首先从与正文各章类似的单序列文本生成示例开始
- 这里依次将它应用于两个提示词（`["2+2?", "3+3=6?"]`）：


In [ ]:
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream_cache,
)
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
    render_prompt,
)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

for problem in ["2+2?", "3+3=6?"]:
    prompt = render_prompt(problem)
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=32,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)

    print()

- 下面使用 `reasoning_from_scratch.qwen3_batched` 中支持批处理的类似代码
- 请注意，批处理版本不支持流式输出，因此必须等所有结果生成完毕后才能解码和输出
- 此处的批量生成使用左侧填充，下一节会对此进行说明
- 在深入内部原理之前，先通过一个使用示例了解其用法


In [ ]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

- 可以看到，结果与之前完全相同
- 区别在于，这些结果通过 `generate_text_basic_batched_cache` 并行生成
- 下一节简要说明其底层工作原理


- 进一步优化的实现会用 `generate_text_basic_batched_cache_stop` 替换 `generate_text_basic_batched_cache`
- `generate_text_basic_batched_cache` 在每个解码步骤中都把所有行保留在活跃批次内
- `generate_text_basic_batched_cache_stop` 会从活跃计算批次中移除已完成的行（内部实现更复杂，但可以优化性能）
- 下图对此进行了说明


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F02_raschka.webp?1" width="500px">

- 补充说明：Qwen3 的 `<eos>` 词元是 `<|endoftext|>`，但为了图示简洁，下图使用 `<eos>`。

&nbsp;
## E.3 填充和注意力掩码

- 在单样本模式下，如果对 `"2+2?"` 这样的短提示词进行分词，可以将其作为形状为 `(1, 4)` 的简单张量传给模型：
  - `input_ids = torch.tensor([[17, 10, 17, 30]])`


- 模型内部会构建标准因果注意力掩码，使每个位置只能关注自身及之前的词元
- 如果不熟悉自注意力，可参阅这篇背景文章：https://magazine.sebastianraschka.com/p/understanding-and-coding-self-attention
- 从概念上说，该掩码如下所示：


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F03_raschka.webp" width="400px">

- `1` 表示“被屏蔽”，`0` 表示“允许”
- 因此第一个词元无法向前查看后续位置，第二个词元只能查看前两个位置，依此类推
- 这是标准的自回归掩码模式


- 批处理改变了这种情况，因为不同提示词通常具有不同长度
- 假设将 `"2+2?"` 与稍长的提示词 `"3+3=6?"` 一起处理
- PyTorch 张量必须为矩形，因此较短的一行必须填充到与较长的一行相同长度
- 这里使用左侧填充：


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F04_raschka.webp" width="500px">

- 请注意，内部还维护了一个额外的 `attn_mask`，它仅用于跟踪填充位置
- 在这个 `attn_mask` 中，`True` 表示已填充，`False` 表示未填充
- 使用额外的 `attn_mask` 识别因果掩码中与填充词元 ID 对应的词元
- 屏蔽填充键并将填充查询置零，是让批处理行为接近单样本执行的重要步骤


- 顺便说明，这里使用 `<|endoftext|>` 词元，但它实际是什么并不重要，因为对应的词元位置无论如何都会被忽略。

In [ ]:
print(tokenizer.pad_token_id)

In [ ]:
print(tokenizer.decode([151643]))

&nbsp;
## E.4 第 3 章：批处理 MATH-500 评估

- 附加材料提供了第 3 章所实现评估方法的脚本，可以像第 6 章那样下载并使用：

In [ ]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)
download_from_github(
    "ch03/01_main-chapter-code/math500_test.json",
    out="math500_test.json",
)

- 随后在代码终端中执行以下命令（如果不使用 uv，请将 `uv run` 替换为 `python`）：

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --which_model "reasoning"
```

- 附加材料还包含应用前述批处理方法的批量生成版本
- 下载方式与之前类似，只需将 `evaluate_math500.py` 替换为 `evaluate_math500_batched.py`


In [ ]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500_batched.py"
)

- 用法也与非批处理版本类似，只是现在额外提供 `--batch_size` 参数，指定 LLM 应并行处理多少个提示词和答案。

```bash
uv run evaluate_math500_batched.py \
  --dataset_size 500 \
  --which_model "reasoning" \
  --batch_size 64
```

- 理想的批量大小取决于硬件承载能力；批量大小为 64 时约使用 23.39 GB 内存（非批处理脚本约使用 1.84 GB）
- 本附录末尾将比较并讨论性能差异


&nbsp;
## E.5 第 4 章：批处理自洽采样

- 实现第 4 章自洽采样的可选脚本 `self_consistency_math500_batched.py` 不会把不同提示词混入同一个填充张量
- 它会将相同提示词重复 `num_samples` 次，并行采样多个续写，再进行自洽投票
- 每一行都从相同的提示词长度开始，因此无需填充；该脚本使用 `reasoning_from_scratch.qwen3` 中的常规 `Qwen3Model`，而不是 `reasoning_from_scratch.qwen3_batched`


- 可以按如下方式下载脚本：

In [ ]:
download_from_github(
    "ch04/02_math500-inference-scaling-scripts/self_consistency_math500_batched.py"
)

- 要下载非批处理版本，只需去掉上述文件名中的 `"_batched"`
- 可以按如下方式运行脚本（非批处理脚本的语法相同）


```bash
uv run self_consistency_math500_batched.py \
  --which_model base \
  --temperature 0.9 \
  --top_p 0.9 \
  --num_samples 3 \
  --dataset_size 500 \
  --prompt_suffix "\n\nExplain step by step."
```

- 本附录末尾将进一步讨论性能。

&nbsp;
## E.6 第 6 章：批处理 GRPO rollout

- 第 5 章的自我改进是一种顺序技术，其本身无法从批处理中受益
- 可以并行运行多个输入的自我改进循环，但实现并不简单，因此未包含在附加材料中
- 接下来改为介绍第 6 章 RLVR 的批处理版本
- 第 6 章对不同 rollout 使用相同提示词，因此无需填充；与 E.5 节类似，代码使用 `reasoning_from_scratch.qwen3` 中的常规 `Qwen3Model` 类
- 可以通过以下方式获取相关脚本：


In [ ]:
# 非批处理版本
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

# 批处理版本
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched.py"
)

# 支持 GPU 的批处理版本
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched_fsdp.py"
)

```bash
uv run rlvr_grpo_original_no_kl_batched.py \
  --num_rollouts 8 \
  --steps 100 \
  --batch_size 4 \
  --max_new_tokens 1024
```

- 当前脚本中的 `--batch_size` 控制一个步骤内并行生成的 rollout 数量
- 这会提高吞吐量，但也会增加内存压力，因此实际使用时可能需要减小 `--num_rollouts` 或 `--max_new_tokens`
- 如果有多张 GPU，FSDP 变体遵循相同模式，并增加 `--num_gpus`
- 同样，本附录末尾会继续讨论性能
- 撰写本文时，附加材料尚未提供第 7 章脚本的批处理版本，但会逐步加入；从概念上说，它们的工作方式与第 6 章脚本类似


&nbsp;
## E.7 第 8 章：批处理蒸馏

- 第 8 章重新采用第 3 章中感知填充的方式，因为蒸馏样本具有不同的提示词和答案长度
- 可以按如下方式下载脚本和样本训练数据集：


In [ ]:
from reasoning_from_scratch.ch08 import load_distill_data

download_from_github(
    "ch08/04_train_with_distillation/distill_batched.py"
)
_ = load_distill_data(
    partition="deepseek-r1-math-train",
    local_path="deepseek-r1-math-train.json",
)

- 对于非批处理版本，去掉文件名中的 `"_batched"`
- 可以按如下方式运行脚本：


```bash
uv run distill_batched.py \
  --data_path deepseek-r1-math-train.json \
  --dataset_size 12000 \
  --validation_size 10 \
  --epochs 2 \
  --use_think_tokens \
  --max_seq_len 1024 \
  --batch_size 4
```

&nbsp;
## E.8 单序列生成与批量生成对比

- 下表汇总了上述脚本的运行时间和内存用量。

| 行 | 脚本                                     | 批量大小 | 内存      | H100 总耗时（分钟） | DGX Spark 总耗时（分钟） |
|-----|------------------------------------------|------------|----------|------------------------|-----------------------------|
| 1   | evaluate_math500.py                      | -          | 1.8 GB   | 90.0                   | 174.7                       |
| 2   | evaluate_math500_batched.py              | 64         | 23.39 GB | 16.0                   | 108.4                       |
|     |                                          |            |          |                        |                             |
| 3   | self_consistency_math500.py              | -          | 1.79 GB  | 252.0                  | 340.8                       |
| 4   | self_consistency_math500_batched.py      | 3          | 2.45 GB  | 129.0                  | 243.3                       |
|     |                                          |            |          |                        |                             |
| 5   | rlvr_grpo_original_no_kl.py              | -          | 43.35 GB | 68.0                   | 63.7                        |
| 6   | rlvr_grpo_original_no_kl_batched.py      | 4          | 44.91 GB | 19.0                   | 23.1                        |
|     |                                          |            |          |                        |                             |
| 7   | distill.py                               | -          | 8.29 GB  | 10.9                   | 32.8                        |
| 8   | distill_batched.py                       | 4          | 8.34 GB  | 9.1                    | 28.2                        |
